# 🗺️ Notebook 07: Seasonal Water Stress Mapping

**Objective:** Generate seasonal water stress zone maps and cluster regions by drought risk profile.

**Contents:**
1. Seasonal water stress patterns
2. K-Means clustering of drought profiles
3. Interactive Folium maps
4. Temporal animation of water stress

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

import config
from src.visualization.maps import (
    create_global_choropleth, create_folium_map, create_seasonal_comparison
)

plt.style.use('dark_background')
print('✅ Setup complete')

In [ ]:
df = pd.read_csv(config.CONSOLIDATED_CSV)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month

print(f'Shape: {df.shape}')
print(f'Seasons: {df["season"].value_counts().to_dict() if "season" in df.columns else "N/A"}')

## 1. Seasonal Water Stress Patterns

In [ ]:
# Seasonal average maps
if 'season' in df.columns and 'drought_composite_score' in df.columns:
    for season in ['DJF', 'MAM', 'JJA', 'SON']:
        season_data = df[df['season'] == season]
        if len(season_data) > 0:
            country_avg = season_data.groupby(['country_code', 'country']).agg({
                'drought_composite_score': 'mean'
            }).reset_index()
            
            season_names = {'DJF': 'Winter (Dec-Feb)', 'MAM': 'Spring (Mar-May)',
                            'JJA': 'Summer (Jun-Aug)', 'SON': 'Autumn (Sep-Nov)'}
            
            fig = px.choropleth(
                country_avg, locations='country_code', locationmode='ISO-3',
                color='drought_composite_score', hover_name='country',
                color_continuous_scale='RdYlBu_r', range_color=[0, 4],
                title=f'Drought Composite Score — {season_names.get(season, season)}',
            )
            fig.update_layout(
                geo=dict(showframe=False, projection_type='natural earth'),
                template='plotly_dark', height=400,
            )
            fig.show()

In [ ]:
# Seasonal comparison for top-risk countries
if 'drought_composite_score' in df.columns:
    top_countries = (df.groupby('country_code')['drought_composite_score']
                     .mean().nlargest(12).index.tolist())
    fig = create_seasonal_comparison(df, country_codes=top_countries)
    if fig:
        fig.show()

## 2. K-Means Clustering of Drought Profiles

In [ ]:
# Create country feature vectors for clustering
cluster_features = [c for c in ['water_stress_score', 'drought_risk_score',
                                 'tws_anomaly_cm', 'groundwater_anomaly_cm',
                                 'drought_composite_score', 'seasonal_variability',
                                 'interannual_variability']
                     if c in df.columns]

country_profiles = df.groupby(['country_code', 'country'])[cluster_features].mean().reset_index()
country_profiles = country_profiles.dropna()

print(f'Country profiles: {country_profiles.shape}')
print(f'Features: {cluster_features}')

In [ ]:
# Elbow method to determine optimal k
X_cluster = country_profiles[cluster_features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
                          marker=dict(size=10, color='#667eea'),
                          line=dict(width=2)))
fig.update_layout(title='Elbow Method — Optimal Number of Clusters',
                  xaxis_title='Number of Clusters (k)', yaxis_title='Inertia',
                  template='plotly_dark', height=400)
fig.show()

In [ ]:
# Fit K-Means with k=4 (matching our 4 risk categories)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
country_profiles['cluster'] = kmeans.fit_predict(X_scaled)

# Name clusters by average drought score
cluster_means = country_profiles.groupby('cluster')['drought_composite_score'].mean().sort_values()
cluster_names = {cluster_means.index[i]: name 
                 for i, name in enumerate(['Low Risk Zone', 'Moderate Risk Zone', 
                                           'High Risk Zone', 'Critical Zone'])}
country_profiles['cluster_name'] = country_profiles['cluster'].map(cluster_names)

print('Cluster assignments:')
for name in ['Low Risk Zone', 'Moderate Risk Zone', 'High Risk Zone', 'Critical Zone']:
    countries = country_profiles[country_profiles['cluster_name'] == name]['country'].tolist()
    print(f'\n{name}:')
    print(f'  {', '.join(countries)}')

In [ ]:
# Map of clusters
color_map = {
    'Low Risk Zone': '#2ecc71',
    'Moderate Risk Zone': '#f39c12',
    'High Risk Zone': '#e74c3c',
    'Critical Zone': '#8e44ad',
}

fig = px.choropleth(
    country_profiles, locations='country_code', locationmode='ISO-3',
    color='cluster_name', hover_name='country',
    color_discrete_map=color_map,
    title='Water Stress Risk Zones (K-Means Clustering)',
    category_orders={'cluster_name': ['Low Risk Zone', 'Moderate Risk Zone', 
                                       'High Risk Zone', 'Critical Zone']},
)
fig.update_layout(
    geo=dict(showframe=False, projection_type='natural earth'),
    template='plotly_dark', height=550,
)
fig.show()

In [ ]:
# Radar chart comparing cluster profiles
cluster_summary = country_profiles.groupby('cluster_name')[cluster_features].mean()

fig = go.Figure()
for cluster_name in cluster_summary.index:
    values = cluster_summary.loc[cluster_name].values
    # Normalize to 0-1 for radar
    values_norm = (values - values.min()) / (values.max() - values.min() + 1e-8)
    
    fig.add_trace(go.Scatterpolar(
        r=list(values_norm) + [values_norm[0]],
        theta=[f.replace('_', ' ').title()[:15] for f in cluster_features] + 
              [cluster_features[0].replace('_', ' ').title()[:15]],
        fill='toself', name=cluster_name,
        line=dict(color=color_map.get(cluster_name, '#999')),
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Cluster Profile Comparison (Radar Chart)',
    template='plotly_dark', height=500, showlegend=True,
)
fig.show()

## 3. Interactive Folium Map

In [ ]:
# Create Folium map
try:
    folium_map = create_folium_map(df, value_col='drought_composite_score')
    
    # Save to HTML
    map_path = config.PROJECT_ROOT / 'docs' / 'water_stress_map.html'
    folium_map.save(str(map_path))
    print(f'✅ Interactive map saved to: {map_path}')
    
    # Display in notebook
    folium_map
except Exception as e:
    print(f'Folium map error: {e}')
    print('Install folium: pip install folium')

## 4. Temporal Animation

In [ ]:
# Animated choropleth over time
if 'year' in df.columns and 'drought_composite_score' in df.columns:
    yearly = df.groupby(['country_code', 'country', 'year']).agg({
        'drought_composite_score': 'mean'
    }).reset_index()
    
    fig = px.choropleth(
        yearly, locations='country_code', locationmode='ISO-3',
        color='drought_composite_score', hover_name='country',
        animation_frame='year',
        color_continuous_scale='RdYlBu_r', range_color=[0, 4],
        title='Drought Composite Score Evolution (2002–2024)',
    )
    fig.update_layout(
        geo=dict(showframe=False, projection_type='natural earth'),
        template='plotly_dark', height=550,
    )
    fig.show()

In [ ]:
print('\n✅ Seasonal Water Stress Mapping Complete')
print(f'Clusters identified: {len(cluster_names)}')
print(f'Countries analyzed: {len(country_profiles)}')